# Lesson 02 — 让一次按下只算一次

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


机械按钮按下和松开时会短暂抖动。程序如果把每一次电平变化都当成新事件，提醒器会误触发。

## 最小知识

去抖（debounce）不是“等待一会儿”这么简单。记录原始电平变化的时间；只有一个电平连续稳定达到窗口长度，才把它当成稳定状态。这里的窗口是 30 ms。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 02-reliable-button guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& board):board_(board){} void tick(){ bool p=board_.read_button()==Level::low; if(p&&!last_){++board_.output.events; board_.output.led=!board_.output.led;} last_=p; } private: Device& board_; bool last_=false; }; }


## 2. 修改一次

把稳定窗口从 `30` 改成 `80`。快速轻按会发生什么？它提高了抗抖动能力，也提高了响应延迟。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 02-reliable-button modify
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& board):board_(board){} void tick(){ bool p=board_.read_button()==Level::low; if(p&&!last_){++board_.output.events; board_.output.led=!board_.output.led;} last_=p; } private: Device& board_; bool last_=false; }; }


## 3. 故障实验（Debug）

下面代码只检测原始边沿。抖动序列会被当作多次按下。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 02-reliable-button broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& board):board_(board){} void tick(){ bool p=board_.read_button()==Level::low; if(p&&!last_){++board_.output.events;board_.output.led=!board_.output.led;} last_=p;} private: Device& board_; bool last_=false; }; }


## 4. 你来完成（Build）

实现一个 30 ms 的稳定窗口。上电时按住按钮只能建立 baseline，释放后再次按下才产生事件。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/02-reliable-button/hints.md)。


In [ ]:
%%academy_lab 02-reliable-button exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& board):board_(board){} void tick(){ const bool pressed=board_.read_button()==Level::low; // TODO: 只在稳定按下时计数
board_.output.state=pressed?"checking":"released"; } private: Device& board_; }; }


## 5. 开放挑战

试着增加“按住 1 秒是长按”的事件。先写清楚短按、长按和抖动之间的优先级。

## 你刚刚真正学到了什么？

你把原始输入与可信事件分开了。时间不是附加功能，而是输入可靠性的一部分。

## 硬件迁移

STEP 2 仍保留这个 Controller；只把 `Device` 换为读 GPIO 和单调时钟的适配器。真实按键的抖动波形与 30 ms 是否合适要用硬件验证。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/02-reliable-button-solution.ipynb)。
